In [0]:
from pyspark.sql.functions import col, avg, max, min, count, desc, round

# --- 1. LEER TABLAS PLATA ---
print("Leyendo capa Plata...")
df_fact = spark.read.table("plata_fact_calidad_aire")
df_dim_tiempo = spark.read.table("plata_dim_tiempo")
df_dim_ciudad = spark.read.table("plata_dim_ciudad")

# Join maestro
df_master = df_fact \
    .join(df_dim_tiempo, "Fecha", "inner") \
    .join(df_dim_ciudad, "Ciudad", "inner")

# --- 2. GENERAR KPI 1: TENDENCIA MENSUAL (oro_kpi_mensual) ---
print("Calculando KPI Mensual...")

df_oro_mensual = df_master.groupBy("Ciudad", "Estado", "Año", "Mes", "Trimestre") \
    .agg(
        round(avg("AQI"), 2).alias("AQI_Promedio"),
        round(avg("PM2_5"), 2).alias("PM2_5_Promedio"),
        max("AQI").alias("AQI_Maximo"),
        count("Fecha").alias("Total_Mediciones")
    ) \
    .orderBy("Año", "Mes", "Ciudad")

# --- 3. GENERAR KPI 2: DÍAS CRÍTICOS (oro_top_criticos) ---
print("Calculando Ranking de Días Críticos...")

# Filtramos AQI > 200 (Malo/Severo)
df_oro_criticos = df_master.filter(col("AQI") > 200) \
    .groupBy("Ciudad", "Estado", "Año") \
    .agg(
        count("Fecha").alias("Dias_Criticos_High_Risk"),
        round(avg("AQI"), 2).alias("AQI_Promedio_En_Crisis")
    ) \
    .orderBy(col("Dias_Criticos_High_Risk").desc())

# --- 4. GUARDAR EN ORO (Tablas Delta) ---
df_oro_mensual.write.format("delta").mode("overwrite").saveAsTable("oro_kpi_mensual")
df_oro_criticos.write.format("delta").mode("overwrite").saveAsTable("oro_top_criticos")

print("✅ Tablas ORO creadas EXITOSAMENTE:")
print("   1. oro_kpi_mensual")
print("   2. oro_top_criticos")

Leyendo capa Plata...
Calculando KPI Mensual...
Calculando Ranking de Días Críticos...
✅ Tablas ORO creadas EXITOSAMENTE:
   1. oro_kpi_mensual
   2. oro_top_criticos


In [0]:
# --- VALIDACIÓN CAPA ORO ---

def mostrar_top_oro(nombre_tabla):
    print(f"\n--- 🏆 Muestra de Datos: {nombre_tabla} ---")
    try:
        df = spark.read.table(nombre_tabla)
        count = df.count()
        print(f"Registros totales: {count}")
        if count > 0:
            df.show(5)
        else:
            print("⚠️ La tabla existe pero está vacía (¿Quizás no hay días con AQI > 200?)")
    except Exception as e:
        print(f"❌ Error: {str(e)}")

# Verificamos
mostrar_top_oro("oro_kpi_mensual")
mostrar_top_oro("oro_top_criticos")


--- 🏆 Muestra de Datos: oro_kpi_mensual ---
Registros totales: 1005
+---------+----------+----+---+---------+------------+--------------+----------+----------------+
|   Ciudad|    Estado| Año|Mes|Trimestre|AQI_Promedio|PM2_5_Promedio|AQI_Maximo|Total_Mediciones|
+---------+----------+----+---+---------+------------+--------------+----------+----------------+
|Ahmedabad|   Gujarat|2015|  1|        1|        33.9|         10.67|     514.0|              31|
|Bengaluru| Karnataka|2015|  1|        1|         0.0|           0.0|       0.0|              31|
|  Chennai|Tamil Nadu|2015|  1|        1|         0.0|           0.0|       0.0|              31|
|    Delhi|     Delhi|2015|  1|        1|      342.29|        175.69|     472.0|              31|
|Hyderabad| Telangana|2015|  1|        1|         0.0|           0.0|       0.0|              28|
+---------+----------+----+---+---------+------------+--------------+----------+----------------+
only showing top 5 rows

--- 🏆 Muestra de Datos: 